# Full LLM generation với Qwen trên Google Colab

Notebook độc lập này sinh câu trả lời cho **toàn bộ** `bm25_bge_top10.jsonl` bằng `Qwen/Qwen2.5-7B-Instruct` trên GPU Colab. Không cần clone repo hay gọi script bên ngoài.

Điểm chính:
- dùng mô hình 4-bit để vừa VRAM của T4;
- ưu tiên `reranked_candidates` (top BGE), chỉ fallback sang `candidates`;
- ghi từng kết quả vào Google Drive và tự resume sau khi Colab ngắt;
- kiểm tra số lượng, ID trùng/thiếu và cho tải kết quả cuối.

> Trước khi chọn **Runtime → Run all**, chọn **Runtime → Change runtime type → T4 GPU**. Lần chạy đầu cần cấp quyền Drive và upload file input khi notebook yêu cầu.

## 1. Cài thư viện

In [ ]:
%pip install -q -U "transformers>=4.45,<5" "accelerate>=0.34" "bitsandbytes>=0.43" sentencepiece tqdm

## 2. Cấu hình GPU, Drive và input

Kết quả được lưu tại `MyDrive/TextMining_RAG_Generation/answers_qwen_colab_full.jsonl`. Nếu `bm25_bge_top10.jsonl` chưa có trong thư mục này, cell sẽ mở hộp thoại upload và chép file vào Drive cho các lần chạy sau.

In [ ]:
import json
import os
import shutil
from pathlib import Path

import torch
from google.colab import drive, files

assert torch.cuda.is_available(), (
    "Không tìm thấy GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại."
)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
WORK_DIR = Path("/content/drive/MyDrive/TextMining_RAG_Generation")
WORK_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = WORK_DIR / "bm25_bge_top10.jsonl"
OUTPUT_PATH = WORK_DIR / "answers_qwen_colab_full.jsonl"
ERROR_PATH = WORK_DIR / "answers_qwen_colab_errors.jsonl"

if not INPUT_PATH.exists():
    print(f"Chưa có {INPUT_PATH.name}. Hãy chọn file từ máy của bạn...")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Chưa upload file input.")
    uploaded_name = next(iter(uploaded))
    shutil.move(f"/content/{uploaded_name}", INPUT_PATH)

print("Input :", INPUT_PATH)
print("Output:", OUTPUT_PATH)

## 3. Kiểm tra dữ liệu

File chuẩn của dự án có 467 dòng, mỗi dòng có 10 `reranked_candidates`. Cell vẫn chấp nhận số dòng khác, nhưng sẽ dừng nếu schema không hợp lệ.

In [ ]:
def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"JSON lỗi ở dòng {line_number}: {exc}") from exc

input_rows = list(read_jsonl(INPUT_PATH))
input_ids = [str(row.get("qa_id")) for row in input_rows]
assert input_rows, "File input rỗng."
assert all(row.get("qa_id") is not None and row.get("question") for row in input_rows), (
    "Mỗi dòng phải có qa_id và question."
)
assert len(input_ids) == len(set(input_ids)), "Input có qa_id trùng nhau."
assert all(row.get("reranked_candidates") or row.get("candidates") for row in input_rows), (
    "Mỗi dòng phải có reranked_candidates hoặc candidates."
)

reranked_count = sum(bool(row.get("reranked_candidates")) for row in input_rows)
print(f"Hợp lệ: {len(input_rows)} QA; {reranked_count} QA có reranked_candidates.")
print("Câu hỏi mẫu:", input_rows[0]["question"])

## 4. Tải Qwen 7B ở chế độ 4-bit

Lần đầu Hugging Face tải khoảng vài GB. Model public nên không cần token. Nếu T4 hết VRAM, giảm `MAX_INPUT_TOKENS` xuống `8192`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
TOP_N_CONTEXT = 5
MAX_INPUT_TOKENS = 16_384
MAX_NEW_TOKENS = 512

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()
print("Đã tải model:", MODEL_ID)

## 5. Hàm tạo prompt và sinh câu trả lời

Context được thêm theo thứ tự rerank. Nếu prompt vượt giới hạn, notebook bỏ dần context hạng thấp; context hạng 1 luôn được giữ.

In [ ]:
SYSTEM_PROMPT = """Bạn là hệ thống hỏi đáp dựa trên tin tức tiếng Việt.
Chỉ trả lời dựa trên context được cung cấp, không suy đoán hay dùng kiến thức ngoài context.
Nếu context không đủ, trả lời đúng câu: Không đủ thông tin trong dữ liệu được cung cấp.
Trả lời ngắn gọn, rõ ràng và đúng trọng tâm."""

def select_candidates(row, top_n=TOP_N_CONTEXT):
    # Quan trọng: ưu tiên kết quả đã rerank bằng BGE.
    candidates = row.get("reranked_candidates") or row.get("candidates") or []
    return candidates[:top_n]

def format_candidate(candidate, position):
    score = candidate.get("rerank_score")
    if score is None:
        score = candidate.get("score", candidate.get("retrieval_score", "N/A"))
    return (
        f"[Context {position}]\n"
        f"Article ID: {candidate.get('article_id')}\n"
        f"Chunk ID: {candidate.get('chunk_id')}\n"
        f"Score: {score}\n"
        f"Text:\n{candidate.get('text', '')}"
    )

def make_messages(question, candidates):
    contexts = "\n\n".join(
        format_candidate(candidate, index)
        for index, candidate in enumerate(candidates, start=1)
    )
    user_prompt = f"Câu hỏi:\n{question}\n\nContext:\n{contexts}\n\nCâu trả lời:"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

def prepare_inputs(question, candidates):
    kept = list(candidates)
    while kept:
        encoded = tokenizer.apply_chat_template(
            make_messages(question, kept),
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        if encoded.shape[-1] <= MAX_INPUT_TOKENS or len(kept) == 1:
            if encoded.shape[-1] > MAX_INPUT_TOKENS:
                raise ValueError(
                    f"Context hạng 1 quá dài ({encoded.shape[-1]} tokens); "
                    "hãy tăng MAX_INPUT_TOKENS hoặc rút gọn input."
                )
            return encoded.to(model.device), kept
        kept.pop()
    raise ValueError("Không có context để generation.")

@torch.inference_mode()
def generate_answer(question, candidates):
    input_ids, used_candidates = prepare_inputs(question, candidates)
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    new_tokens = output_ids[0, input_ids.shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, used_candidates

## 6. Smoke test 1 câu

Cell này không ghi vào output. Kiểm tra câu trả lời và VRAM trước khi chạy full.

In [ ]:
sample = input_rows[0]
sample_answer, sample_contexts = generate_answer(
    sample["question"], select_candidates(sample)
)
print("Q:", sample["question"])
print("A:", sample_answer)
print("Số context đã dùng:", len(sample_contexts))

## 7. Chạy full và checkpoint từng QA

Có thể chạy lại cell này bất kỳ lúc nào. Những `qa_id` đã có output hợp lệ sẽ được bỏ qua. Lỗi của từng QA được ghi riêng và QA đó sẽ được thử lại ở lần chạy sau. Trên T4, 467 câu có thể mất nhiều giờ và có thể cần nhiều phiên Colab.

In [ ]:
from datetime import datetime, timezone
from tqdm.auto import tqdm

def repair_checkpoint(path):
    path = Path(path)
    if not path.exists():
        return set()

    valid_rows = []
    done = set()
    needs_repair = False
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            try:
                row = json.loads(line)
                qa_id = row.get("qa_id")
                if qa_id is None or not row.get("generated_answer"):
                    needs_repair = True
                    continue
                qa_id = str(qa_id)
                if qa_id in done:
                    needs_repair = True
                    continue
                valid_rows.append(row)
                done.add(qa_id)
            except json.JSONDecodeError:
                # Có thể xảy ra nếu runtime ngắt giữa lúc ghi một dòng.
                needs_repair = True

    if needs_repair:
        backup_path = path.with_suffix(path.suffix + ".backup")
        shutil.copy2(path, backup_path)
        temporary_path = path.with_suffix(path.suffix + ".tmp")
        with temporary_path.open("w", encoding="utf-8") as handle:
            for row in valid_rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        temporary_path.replace(path)
        print("Đã sửa checkpoint; bản cũ lưu tại:", backup_path)
    return done

done_ids = repair_checkpoint(OUTPUT_PATH)
print(f"Resume: đã có {len(done_ids)}/{len(input_rows)} QA.")

with OUTPUT_PATH.open("a", encoding="utf-8") as output_file, ERROR_PATH.open(
    "a", encoding="utf-8"
) as error_file:
    progress = tqdm(input_rows, total=len(input_rows), desc="Qwen generation")
    for row in progress:
        qa_id = str(row["qa_id"])
        if qa_id in done_ids:
            continue

        try:
            selected = select_candidates(row)
            answer, used_contexts = generate_answer(row["question"], selected)
            if not answer:
                raise ValueError("Model trả về câu rỗng.")

            output_row = {
                "qa_id": row.get("qa_id"),
                "qa_type": row.get("qa_type"),
                "question": row.get("question"),
                "strategy": row.get("strategy"),
                "is_possible": row.get("is_possible"),
                "llm_model": MODEL_ID,
                "backend": "transformers-4bit",
                "top_n_context": len(used_contexts),
                "generated_answer": answer,
                "gold_chunk_ids": row.get("gold_chunk_ids", []),
                "used_contexts": used_contexts,
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            output_file.write(json.dumps(output_row, ensure_ascii=False) + "\n")
            output_file.flush()
            done_ids.add(qa_id)
        except Exception as exc:
            error_row = {
                "qa_id": row.get("qa_id"),
                "error": repr(exc),
                "failed_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            error_file.write(json.dumps(error_row, ensure_ascii=False) + "\n")
            error_file.flush()
            print(f"\nLỗi qa_id={qa_id}: {exc}")

        progress.set_postfix(done=f"{len(done_ids)}/{len(input_rows)}")

print(f"Hoàn tất hiện tại: {len(done_ids)}/{len(input_rows)} QA")
print("Checkpoint:", OUTPUT_PATH)

## 8. Kiểm tra kết quả và tải file

Chỉ tải file khi `missing = 0` và `duplicates = 0`. Nếu còn thiếu, chạy lại cell full generation.

In [ ]:
from collections import Counter

output_rows = list(read_jsonl(OUTPUT_PATH)) if OUTPUT_PATH.exists() else []
output_ids = [str(row.get("qa_id")) for row in output_rows]
id_counts = Counter(output_ids)
duplicate_ids = sorted(qa_id for qa_id, count in id_counts.items() if count > 1)
missing_ids = sorted(set(input_ids) - set(output_ids))
unexpected_ids = sorted(set(output_ids) - set(input_ids))

print("Input rows    :", len(input_rows))
print("Output rows   :", len(output_rows))
print("Unique output :", len(set(output_ids)))
print("Missing       :", len(missing_ids), missing_ids[:10])
print("Duplicates    :", len(duplicate_ids), duplicate_ids[:10])
print("Unexpected    :", len(unexpected_ids), unexpected_ids[:10])

if output_rows:
    print("\n--- Kết quả mẫu ---")
    for row in output_rows[:3]:
        print("Q:", row["question"])
        print("A:", row["generated_answer"])
        print("-" * 70)

assert not missing_ids, "Còn QA chưa sinh. Chạy lại cell full generation để resume."
assert not duplicate_ids, "Output có qa_id trùng; cần làm sạch trước khi dùng."
assert not unexpected_ids, "Output có qa_id không thuộc input."
print("\nKết quả đầy đủ và hợp lệ.")
files.download(str(OUTPUT_PATH))